In [2]:
import ee
import json

import ipyleaflet
from ipywidgets import HTML
from ipyleaflet import TileLayer
from shapely.geometry import shape
from ipyleaflet import Marker, DivIcon

import matplotlib.pyplot as plt

from landcover_explorer.settings import Settings
from landcover_explorer.knowledgebase.gee_tiles_preprocess import (
    load_image,
    assign_biomass_risk_score,
    assign_forest_type_risk_score,
    load_recent_forest_loss,
    compute_distance_to_loss,
    assign_distance_risk_score,
    aggregate_for_resampling,
    loss_year_window
)

settings = Settings()

In [3]:
select_year = 2020
select_iso = "MYS"

In [4]:
credentials = ee.ServiceAccountCredentials(
    settings.google_earth_service_account, str(settings.google_earth_key)
)
ee.Initialize(credentials)

In [5]:
with open(settings.gadm_path) as f:
    gadm = json.load(f)

iso_feature = next(f for f in gadm["features"] if f["properties"]["GID_0"] == select_iso)
geometry = ee.Geometry(iso_feature["geometry"])

centroid = geometry.centroid().getInfo()
coords = centroid['coordinates']  # [longitude, latitude]
center = [coords[1], coords[0]]  # ipyleaflet uses [lat, lon]

In [6]:
# FAO GAUL admin1 (state/province) boundaries, restricted to the country of interest by
# name (GADM's COUNTRY property) rather than geometry, since GAUL and GADM boundaries
# don't align exactly and filterBounds could pull in slivers of neighboring countries.
admin1 = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(
    ee.Filter.eq("ADM0_NAME", iso_feature["properties"]["COUNTRY"])
)

# Paint boundary outlines only (no fill) so the risk score layers underneath stay visible.
admin1_outline = ee.Image().byte().paint(
    featureCollection=admin1,
    color=1,
    width=2,
)

admin1_tile = admin1_outline.getMapId({"palette": ["000000"]})

In [7]:
forest_dataset = load_image(settings.modis_collection_id, settings.modis_collection_band_name, select_year, geometry)
forest_dataset = assign_forest_type_risk_score(forest_dataset)

target_projection = forest_dataset.projection() # Sinusoidal: SR-ORG:6974
target_scale = target_projection.nominalScale() # 463.31 meters

In [8]:
# AGB (100m)
agb_100m = load_image(settings.biomass_collection_id, settings.biomass_collection_band_name, select_year, geometry)
agb_aligned = aggregate_for_resampling(agb_100m)
agb_risk_score = assign_biomass_risk_score(agb_aligned)

In [9]:
# agb_risk_score_stats = agb_risk_score.reduceRegion(
#     reducer=ee.Reducer.minMax(),
#     geometry=geometry,
#     scale=target_scale,
#     bestEffort=True,
#     maxPixels=1e9
# ).getInfo()

In [10]:
# forest_dataset_stats = forest_dataset.reduceRegion(
#     reducer=ee.Reducer.minMax(),
#     geometry=geometry,
#     scale=settings.modis_collection_resolution,
#     bestEffort=True,
#     maxPixels=1e9
# ).getInfo()

---

## Calculate distance to forest loss

In [11]:
# Hansen binary forest loss (30m), clipped to the country boundary, filtered to
# select_year's 5-year loss-year bucket (e.g. 2018 -> 2016-2020), and restricted to
# pixels that were forest (forest_dataset).
loss_recent = load_recent_forest_loss(select_year, geometry=geometry, forest_mask=forest_dataset)

In [12]:
# Distance transform over the country-clipped loss mask, clipped again to geometry since
# fastDistanceTransform doesn't inherit loss_recent's footprint. Note: pixels near the
# border may undercount distance since loss just outside the border isn't considered.
distance_m = compute_distance_to_loss(loss_recent, geometry=geometry)

# Reclassify to a 0-3 risk score, then align to forest_dataset's resolution.
distance_risk_score = assign_distance_risk_score(distance_m)
distance_risk_score_aligned = aggregate_for_resampling(distance_risk_score)

In [13]:
distance_tile = distance_risk_score.getMapId(
    {
        "min": 0,
        "max": 3,
        "palette": ["ffffff", "ffff00", "ff8c00", "ff0000"],
    }
)

m = ipyleaflet.Map(center=center, zoom=6)
m.add_layer(
    ipyleaflet.TileLayer(
        url=distance_tile["tile_fetcher"].url_format,
        name="Distance Risk Score",
        opacity=0.7,
    )
)
m.add_control(ipyleaflet.LayersControl())
m

Map(center=[3.81519137212966, 109.71312288596678], controls=(ZoomControl(options=['position', 'zoom_in_text', …

In [14]:
loss_window_min, loss_window_max = loss_year_window(select_year)
distance_asset_id = f"projects/{settings.google_earth_project_id}/assets/distance_risk_score_{select_iso}_{loss_window_min}_{loss_window_max}"

try:
    ee.data.getAsset(distance_asset_id)
    print(f"Reusing existing asset for {select_iso} window {loss_window_min}-{loss_window_max}: {distance_asset_id}")
    distance_export_task = None
except ee.EEException:
    distance_export_task = ee.batch.Export.image.toAsset(
        image=distance_risk_score_aligned,
        description=f"distance_risk_score_{select_iso}_{loss_window_min}_{loss_window_max}",
        assetId=distance_asset_id,
        region=geometry,
        scale=target_scale,
        maxPixels=1e10,
    )
    distance_export_task.start()

distance_export_task.status()["state"] if distance_export_task else "ASSET_EXISTS"

Reusing existing asset for MYS window 16-20: projects/forest-health-dashboard/assets/distance_risk_score_MYS_16_20


'ASSET_EXISTS'

In [15]:
import time

if distance_export_task is not None:
    while distance_export_task.active():
        print(distance_export_task.status()["state"])
        time.sleep(30)
    print(distance_export_task.status())
else:
    print("Skipped export; using existing asset.")

Skipped export; using existing asset.


---

## Calculate aggregated risk score

In [16]:
# Read back the cached distance risk score for this loss-year window from the GEE asset
# instead of recomputing the fastDistanceTransform chain. Already bounded to geometry
# since the export in the previous cell used region=geometry.
distance_risk_score_aligned = ee.Image(distance_asset_id) # Will be handled in app.py

# Aggregate risk score: weighted sum of AGB, forest presence, and proximity to recent loss.
# Risk = 0.4*AGB + 0.2*ForestDataset + 0.4*DistanceRisk
aggregate_risk_score = (
    agb_risk_score.multiply(0.4)
    .add(forest_dataset.multiply(0.2))
    .add(distance_risk_score_aligned.multiply(0.4))
)

In [17]:
# No batch export needed here: the expensive fastDistanceTransform work is already
# cached as distance_asset_id, and combining it with AGB/forest (both 463m-native) is
# a cheap pixel-wise op, so an interactive reduceRegion is fine.
aggregate_risk_score_stats = aggregate_risk_score.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=geometry,
    scale=target_scale,
    bestEffort=True,
    maxPixels=1e9,
).getInfo()
aggregate_risk_score_stats

{'constant_max': 3.0000000000000004, 'constant_min': 0.9842183992006466}

---
## Rank admin1 regions by highest-risk forest area

In [18]:
# round() guards against floating-point noise (aggregate_risk_score_stats showed a max
# of 3.0000000000000004 rather than exactly 3.0).
risk3_mask = aggregate_risk_score.round().eq(3)
risk3_area_m2 = risk3_mask.multiply(ee.Image.pixelArea())

admin1_risk3_area = risk3_area_m2.reduceRegions(
    collection=admin1,
    reducer=ee.Reducer.sum().setOutputs(["risk3_area_m2"]),
    scale=target_scale,
    tileScale=4,
)

In [19]:
admin1_risk3_area_ranked = admin1_risk3_area.sort("risk3_area_m2", False)

admin1_risk3_area_ranked_info = admin1_risk3_area_ranked.getInfo()["features"]
for rank, feature in enumerate(admin1_risk3_area_ranked_info, start=1):
    props = feature["properties"]
    area_km2 = props["risk3_area_m2"] / 1e6
    print(f"{rank}. {props['ADM1_NAME']}: {area_km2:,.2f} km2")

1. Sarawak: 57,201.61 km2
2. Sabah: 19,428.82 km2
3. Pahang: 11,534.95 km2
4. Perak: 5,032.94 km2
5. Kelantan: 4,035.39 km2
6. Terengganu: 3,596.17 km2
7. Johor: 3,152.95 km2
8. Kedah: 1,307.82 km2
9. Negeri Sembilan: 988.41 km2
10. Selangor: 766.71 km2
11. Perlis: 85.47 km2
12. Pulau Pinang: 45.51 km2
13. Melaka: 35.05 km2
14. Labuan: 5.62 km2
15. Kuala Lumpur: 1.23 km2


---

## Visualization

In [20]:
forest_tile = forest_dataset.getMapId(
    {
        "min" : 3,
        "max" : 3,
        "palette": "#05450a"
    }
)

agb_tile = agb_risk_score.getMapId(
    {
        "min" : 0,
        "max" : 3,
        "palette" : ['ffffff', 'ffff00', 'ff8c00', 'ff0000']
    }
)

aggregate_tile = aggregate_risk_score.getMapId(
    {
        "min": 0,
        "max": 3,
        "palette": ["ffffff", "ffff00", "ff8c00", "ff0000", "8b0000"],
    }
)

In [28]:
top5_admin1 = admin1_risk3_area_ranked.limit(5)

# Thick outline (no fill) so the top 5 stand out against admin1_outline.
top5_outline = ee.Image().byte().paint(featureCollection=top5_admin1, color=1, width=4)
top5_highlight = top5_outline.visualize(palette=["ff00ff"])

top5_tile = top5_highlight.getMapId()

In [29]:
top5_info = admin1_risk3_area_ranked_info[:5]
top5_total_km2 = sum(f["properties"]["risk3_area_m2"] for f in top5_info) / 1e6

infobox_rows = "".join(
    f"<tr><td>{rank}. {f['properties']['ADM1_NAME']}</td>"
    f"<td style='text-align:right'>{f['properties']['risk3_area_m2'] / 1e6:,.1f} km2</td></tr>"
    for rank, f in enumerate(top5_info, start=1)
)

infobox_html = f"""
<div style="font-family: sans-serif; font-size: 13px; line-height: 1.4;">
  <b>Top 5 Highest-Risk Admin1 (Forest, Risk = 3)</b>
  <table style="margin-top: 4px;">{infobox_rows}</table>
  <hr style="margin: 4px 0;">
  <b>Total: {top5_total_km2:,.1f} km2</b>
</div>
"""

infobox = HTML(value=infobox_html)
infobox.layout.margin = "10px"
infobox.layout.padding = "8px"
infobox.layout.background = "white"
infobox.layout.border = "1px solid #999"

m = ipyleaflet.Map(center=center, zoom=6)
m.add_layer(
    ipyleaflet.TileLayer(
        url=aggregate_tile["tile_fetcher"].url_format,
        name="Aggregate Risk Score",
        opacity=0.7,
    )
)
m.add_layer(ipyleaflet.TileLayer(url=forest_tile['tile_fetcher'].url_format, name='Forest Risk Score', opacity=0.6))
m.add_layer(ipyleaflet.TileLayer(url=agb_tile['tile_fetcher'].url_format, name='AGB Risk Score', opacity=0.6))

m.add_layer(
    ipyleaflet.TileLayer(
        url=top5_tile["tile_fetcher"].url_format,
        name="Top 5 Highest-Risk Admin1",
        opacity=1,
    )
)

# Rank badges (1-5) placed at each top-5 admin1 region's centroid.
# Leaflet's built-in ".leaflet-div-icon" class puts a white box + border around every
# DivIcon's container by default, independent of the html below, so it must be
# neutralized explicitly or it shows as a square behind the circular badge.
badge_reset_style = "<style>.leaflet-div-icon { background: transparent; border: none; }</style>"

for rank, feature in enumerate(top5_info, start=1):
    centroid = shape(feature["geometry"]).centroid
    badge_icon = DivIcon(
        html=(
            badge_reset_style +
            f'<div style="font-size:14px; font-weight:bold; color:white; '
            f'background:#d6006d; border:2px solid white; border-radius:50%; '
            f'width:24px; height:24px; display:flex; align-items:center; '
            f'justify-content:center;">{rank}</div>'
        ),
        icon_size=[24, 24],
        icon_anchor=[12, 12],
    )
    m.add_layer(Marker(location=(centroid.y, centroid.x), icon=badge_icon, draggable=False))

m.add_control(ipyleaflet.LayersControl())
m.add_control(ipyleaflet.WidgetControl(widget=infobox, position="topright"))
m

Map(center=[3.81519137212966, 109.71312288596678], controls=(ZoomControl(options=['position', 'zoom_in_text', …